# AI-Driven Market Analysis for Computer Component Price Surge

**Decision Support System for Hardware Procurement in the AI Era**

Pipeline: Scraping → Preprocessing → Statistical Analysis → Sentiment → AHP-TOPSIS → Visualization.

> Run cells top to bottom. CPU runtime is sufficient.

## 1. Environment Setup

In [1]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO = "ai-era-pc-component-market-analysis"
    if Path(REPO).exists():
        subprocess.run(["git", "-C", REPO, "pull", "-q"], check=False)
    else:
        subprocess.run(
            [
                "git",
                "clone",
                "-q",
                "https://github.com/bugkey24/ai-era-pc-component-market-analysis.git",
            ],
            check=True,
        )
    get_ipython().run_line_magic("cd", REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    head = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True)
    print("repo commit:", head.stdout.strip())

import nltk

nltk.download("stopwords", quiet=True)
print("Environment ready. IN_COLAB =", IN_COLAB)

Environment ready. IN_COLAB = False


## 2. Configuration & Imports

In [2]:
from src.utils import load_config, setup_logger

CONFIG_PATH = "config.yaml"
config = load_config(CONFIG_PATH)
logger = setup_logger(name="notebook", level=config["logging"]["level"])
config["scraping"]["platforms"]

[{'name': 'tokopedia',
  'enabled': True,
  'method': 'static',
  'base_url': 'https://www.tokopedia.com',
  'reviews_enabled': True,
  'search_keywords': {'gpu': 'rtx', 'ram': 'ddr5', 'ssd': 'nvme'}},
 {'name': 'shopee',
  'enabled': True,
  'method': 'dynamic',
  'base_url': 'https://shopee.co.id',
  'reviews_enabled': True},
 {'name': 'blibli',
  'enabled': True,
  'method': 'static',
  'base_url': 'https://www.blibli.com',
  'reviews_enabled': False}]

## 3. Option A — Load Existing Data

Skip live scraping (platform markup changes often). Place CSVs in `data/raw/` or use the sample generator below.

In [3]:
from pathlib import Path

import pandas as pd

# Real experiment data ships with the repo (data/snapshot/);
# fresher local runs in data/raw take precedence, synthetic sample is the last resort.
candidates = [
    sorted(Path("data/snapshot").glob("*products*.csv")),
    sorted(Path("data/raw").glob("*products*.csv")),
]
csvs = next((c for c in candidates if c), [])

if csvs:
    data = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
    print(f"Loaded {len(data)} rows from {len(csvs)} file(s): {[f.name for f in csvs]}")
else:
    # Minimal sample so the pipeline is runnable end-to-end offline
    import numpy as np

    rng = np.random.default_rng(42)
    rows = []
    for cat, base, n in [("gpu", 8_000_000, 20), ("ram", 1_400_000, 20), ("ssd", 800_000, 20)]:
        for i in range(n):
            rows.append(
                {
                    "product_id": f"{cat.upper()}-{i:03d}",
                    "name": f"Sample {cat.upper()} Model {i} {rng.choice(['8GB', '16GB', '512GB', '1TB'])}",
                    "category": cat,
                    "price": f"Rp {base * rng.uniform(0.6, 1.6):,.0f}",
                    "rating": round(rng.uniform(3.8, 5.0), 1),
                    "review_count": int(rng.integers(5, 400)),
                    "seller_rating": round(rng.uniform(4.0, 5.0), 1),
                    "seller_followers": int(rng.integers(10, 5000)),
                    "source": str(rng.choice(["tokopedia", "shopee", "blibli"])),
                }
            )
    data = pd.DataFrame(rows)
    print(f"Generated {len(data)} sample rows (no snapshot or raw CSVs found)")
data.head()

Loaded 247 rows from 1 file(s): ['tokopedia_products.csv']


,product_id,name,url,price,original_price,discount,rating,review_count,seller_name,seller_tier,location,source,category
0,100702835200,ZOTAC GAMING NVIDIA GeForce RTX 5070 Ti SOLID ...,https://www.tokopedia.com/gamingpcstore/zotac-...,25288000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu
1,100702919487,MSI NVIDIA GEFORCE RTX 5050 8GB VENTUS 2X OC G...,https://www.tokopedia.com/gamingpcstore/msi-nv...,8955000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu
2,100552637943,MANLI STELLAR NVIDIA GeForce RTX 5080 OC 16GB ...,https://www.tokopedia.com/gamingpcstore/manli-...,31900000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu
3,100276456043,ZOTAC GAMING NVIDIA GeForce RTX 5060 Ti 16GB T...,https://www.tokopedia.com/gamingpcstore/zotac-...,15486000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu
4,100493453963,Gainward NVIDIA GeForce RTX 5060 Python III 8G...,https://www.tokopedia.com/gamingpcstore/gainwa...,10730000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu


## 4. Option B — Live Scraping (opt-in)

⚠️ **Expect failures from Colab.** Colab runs on Google datacenter IPs, which platforms throttle or block: Tokopedia times out, Blibli returns 403, Shopee additionally needs a Chrome install. Our validated collections ran from a **residential IP** — see `docs/09-live-experiment-results.md`. The snapshot in cell above is the reliable path.

`try_live_scrape()` below is **guarded**: a 0-row result (the expected Colab outcome) never overwrites your dataset. It stays commented out so *Run all* performs pure analysis — uncomment the last line to opt in.

In [4]:
from src import PipelineOrchestrator  # self-contained: no earlier-cell dependency


def try_live_scrape():
    """Attempt live scraping. A 0-row result NEVER overwrites `data`."""
    pipeline = PipelineOrchestrator(CONFIG_PATH)
    scraped = pipeline._run_scraping()
    if scraped.empty:
        print(
            "Live scraping returned 0 rows (expected from datacenter IPs — "
            "see docs/10 §5). Dataset unchanged."
        )
        return None
    globals()["data"] = scraped
    print(f"Live scrape: {len(scraped)} rows — dataset replaced.")
    return scraped


# try_live_scrape()   # ← uncomment to opt in

## 5. Preprocessing & Feature Engineering

In [5]:
from src.preprocessing import DataPreprocessor, FeatureEngineer

clean = (
    DataPreprocessor(data)
    .clean_prices()
    .handle_missing(config["preprocessing"].get("handle_missing", "drop"))
    .extract_specifications()
    .remove_outliers(threshold=config["preprocessing"].get("outlier_threshold", 3.0))
)
fe = FeatureEngineer(clean.df)
clean = fe.create_price_per_gb().create_weighted_rating().create_seller_trust_score()
df = clean.get_engineered_data()
print(f"{len(df)} rows, {len(df.columns)} columns")
df.head()

240 rows, 18 columns


,product_id,name,url,price,original_price,discount,rating,review_count,seller_name,seller_tier,location,source,category,spec_capacity,spec_memory_type,spec_interface,price_per_gb,weighted_rating
0,100702835200,ZOTAC GAMING NVIDIA GeForce RTX 5070 Ti SOLID ...,https://www.tokopedia.com/gamingpcstore/zotac-...,25288000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu,16GB,,,1580500.0,0.0
1,100702919487,MSI NVIDIA GEFORCE RTX 5050 8GB VENTUS 2X OC G...,https://www.tokopedia.com/gamingpcstore/msi-nv...,8955000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu,8GB,,,1119375.0,0.0
2,100552637943,MANLI STELLAR NVIDIA GeForce RTX 5080 OC 16GB ...,https://www.tokopedia.com/gamingpcstore/manli-...,31900000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu,16GB,,,1993750.0,0.0
3,100276456043,ZOTAC GAMING NVIDIA GeForce RTX 5060 Ti 16GB T...,https://www.tokopedia.com/gamingpcstore/zotac-...,15486000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu,16GB,,,967875.0,0.0
4,100493453963,Gainward NVIDIA GeForce RTX 5060 Python III 8G...,https://www.tokopedia.com/gamingpcstore/gainwa...,10730000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu,8GB,,,1341250.0,0.0


## 6. Statistical Analysis

In [6]:
from src.analysis import StatisticalAnalyzer

stats_an = StatisticalAnalyzer(df)
summary = stats_an.describe().get_summary()
display(summary.round(2))
stats_an.correlation_matrix()
display(stats_an.price_trend_by_category())
stats_an.normality_test("price")

,count,mean,std,min,25%,50%,75%,max,median,iqr,skewness,kurtosis
product_id,240.0,4.985331e+10,4.444118e+10,4.950740e+08,1.257713e+10,1.608577e+10,1.007236e+11,1.037239e+11,1.608577e+10,8.814647e+10,0.31,-1.90
price,240.0,1.021729e+07,8.916398e+06,1.410000e+05,3.704250e+06,8.450000e+06,1.330988e+07,5.273200e+07,8.450000e+06,9.605625e+06,1.73,3.44
original_price,240.0,3.675649e+06,9.318616e+06,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,6.673700e+07,0.000000e+00,0.000000e+00,3.36,13.37
discount,240.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.00,0.00
rating,240.0,4.970000e+00,1.500000e-01,3.500000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,0.000000e+00,-7.42,60.12
review_count,240.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.00,0.00
seller_tier,240.0,2.000000e+00,0.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,0.000000e+00,0.00,0.00
price_per_gb,219.0,6.257182e+05,6.869448e+05,2.585450e+03,4.699220e+03,3.125000e+05,1.094062e+06,3.634833e+06,3.125000e+05,1.089363e+06,1.27,1.76
weighted_rating,240.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.00,0.00


,mean,median,std,min,max,count,cv
category,,,,,,,
gpu,17121967.62,14633000.0,9958484.07,5005000,52732000,94,0.582
ram,7700472.24,8544500.0,4435715.49,1400000,20950000,72,0.576
ssd,3895293.24,3479000.0,2580268.20,141000,13146500,74,0.662


{'statistic': 0.8339, 'p_value': 0.0, 'normal_at_0.05': False}

## 7. Sentiment Analysis

The SVM trains automatically when review CSVs exist in data/raw/reviews_*.csv (produced by the review scrapers; the pipeline merges per-product scores). Labels are weak supervision from review ratings. If the corpus is skewed (e-commerce reality: mostly positive), the model fits on full data and accuracy is reported as not measurable. A demo set below keeps this notebook runnable standalone.

In [7]:
from src.analysis import SentimentAnalyzer

demo_texts = [
    "harga mahal sekali",
    "terlalu mahal untuk spesifikasi ini",
    "harga naik terus",
    "harga murah dan terjangkau",
    "murah bagus worth it",
    "harga oke murah",
    "performa cepat stabil",
    "cepat dan stabil untuk AI training",
    "performa mantap",
    "lambat dan sering hang",
    "performa jelek lambat",
    "lemot sering ngehang",
] * 3
demo_labels = (["negative"] * 3 + ["positive"] * 3 + ["positive"] * 3 + ["negative"] * 3) * 3

sentiment = SentimentAnalyzer(language=config["sentiment"]["language"])
sentiment.train(demo_texts, demo_labels)
print(f"Demo-model accuracy: {sentiment.accuracy:.2f}  (replace with real labelled reviews)")

Demo-model accuracy: 1.00  (replace with real labelled reviews)


## 8. AHP-TOPSIS Decision Model

In [8]:
import numpy as np

from src.dss import AHPProcessor, TOPSISProcessor

dss_cfg = config["dss"]

ahp = AHPProcessor(dss_cfg["criteria"])
ahp.build_pairwise_matrix(dss_cfg["pairwise_matrix"])
ahp.calculate_weights().check_consistency()
print(ahp.summary())
assert ahp.is_consistent(), "Pairwise matrix inconsistent (CR >= 0.1) — revise config"

{'criteria': ['price', 'performance', 'rating', 'seller_reliability', 'sentiment', 'future_value'], 'weights': {'price': np.float64(0.227), 'performance': np.float64(0.4387), 'rating': np.float64(0.0918), 'seller_reliability': np.float64(0.0413), 'sentiment': np.float64(0.0413), 'future_value': np.float64(0.1598)}, 'lambda_max': 6.3633, 'consistency_ratio': 0.0586, 'is_consistent': True}


In [9]:
# Decision matrix: map config criteria to available columns.
# Column availability differs by dataset (live cache has seller_tier but no
# seller_rating/followers; synthetic demo data is the reverse) — resolve
# with fallbacks instead of assuming.
def col_for(criterion: str) -> str:
    candidates = {
        "price": ["price"],
        "performance": ["rating"],
        "rating": ["weighted_rating", "rating"],
        "seller_reliability": ["seller_tier", "seller_trust", "rating"],
        "sentiment": ["sentiment_score", "rating"],
        "future_value": ["price_per_gb"],
    }[criterion]
    return next((c for c in candidates if c in df.columns), candidates[-1])


matrix = np.column_stack(
    [pd.to_numeric(df[col_for(c)], errors="coerce").fillna(0) for c in dss_cfg["criteria"]]
)

topsis = TOPSISProcessor(matrix, ahp.get_weights(), dss_cfg["criteria_types"])
ranking = topsis.rank()
df_ranked = (
    df.reset_index(drop=True)
    .loc[ranking["Alternative"]]
    .assign(Score=ranking["Score"].values, Rank=ranking["Rank"].values)
)
df_ranked[["name", "category", "price", "rating", "Score", "Rank"]].head(10)

,name,category,price,rating,Score,Rank
215,CORSAIR MP600 ELITE 1TB PCIe Gen4 x4 NVMe 1.4 ...,ssd,3949000,5.0,0.5573,1
19,VGA Zotac GeForce RTX 3050 6GB GDDR6 TWIN EDGE OC,gpu,5199000,5.0,0.6190,2
222,KINGSTON KC3000 1024GB 1TB SSD PCIE 4.0 GEN4 M...,ssd,4330000,5.0,0.5552,3
146,Team Elite Plus DDR5 6000MHz Dual Channel 32GB...,ram,9979050,5.0,0.5404,4
8,COLORFUL IGAME NVIDIA GEFORCE RTX 5050 ULTRA W...,gpu,8583000,5.0,0.6163,5
10,VGA PALIT GeForce RTX 5050 StormX 8GB GDDR6,gpu,7499000,5.0,0.6109,6
14,Gainward NVIDIA GeForce RTX 5060 Ti Ghost 8GB ...,gpu,10740000,5.0,0.6269,7
29,ZOTAC GeForce RTX 5060 Ti Twin Edge WHITE 16GB...,gpu,14250000,5.0,0.5527,8
26,Gainward NVIDIA GeForce RTX 5070 Phoenix 12GB ...,gpu,18700000,3.5,0.5575,9
54,Zotac GeForce RTX 5050 8GB GDDR6 Twin Edge OC,gpu,7826000,5.0,0.6126,10


## 9. Visualization

In [10]:
from src.visualization import Visualizer

viz = Visualizer(
    df,
    output_dir="outputs/visualizations",
    **{k: v for k, v in config["visualization"].items() if k in ("style", "palette", "dpi")},
)
_ = viz.plot_price_trends()
_ = viz.plot_correlation_heatmap()
_ = viz.plot_ranking_bar_chart(ranking)
print("Charts saved to outputs/visualizations/")

Charts saved to outputs/visualizations/


## 10. Export Results

In [11]:
from pathlib import Path

out = Path("outputs")
out.mkdir(exist_ok=True)
df.to_csv(out / "cleaned_data.csv", index=False)
ranking.to_csv(out / "rankings.csv", index=False)
print("Saved:", [p.name for p in out.iterdir()])

Saved: ['cleaned_data.csv', 'prediction.json', 'rankings.csv', 'reviews_sample.csv', 'run_summary.json', 'visualizations']


## 11. Conclusions

- **Why prices rose:** fill in after running against live data.
- **Top value pick:** see ranking table above.
- **Normalization outlook:** see `docs/03-methodology.md` Phase 6 scenarios.

*Replace the demo sample/sentiment data with real scraped data and labelled reviews for production-grade results.*